In [1]:
import pandas as pd
import numpy as np
from sklearn import set_config

# Force Scikit-Learn to keep features inside named Pandas DataFrames across the pipeline
set_config(transform_output="pandas")

# Load the raw datasets
train_df = pd.read_csv("datasets/house-prices-advanced-regression-techniques/train.csv")
test_df = pd.read_csv("datasets/house-prices-advanced-regression-techniques/test.csv")

# Filter out the extreme outliers found during EDA
train_df = train_df[train_df['BsmtFinSF1'].fillna(0) < 5000].copy()
train_df = train_df[train_df['GrLivArea'].fillna(0) < 4000].copy()

# Split into features and log-transformed target space
X_train = train_df.drop(columns=['SalePrice', 'Id'], errors='ignore')
y_train = np.log1p(train_df['SalePrice'])

X_test = test_df.drop(columns=['Id'], errors='ignore')
submission_ids = test_df['Id']

print(f"Train features matrix shape: {X_train.shape}")
print(f"Test features matrix shape:  {X_test.shape}")

Train features matrix shape: (1456, 79)
Test features matrix shape:  (1459, 79)


In [17]:
from sklearn.base import BaseEstimator, TransformerMixin

class RealEstateFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass
        
    def fit(self, X, y=None):
        self.is_fitted_ = True
        return self
        
    def transform(self, X):
        df = X.copy()
        
        # Spatial transformations
        df['LotArea_log'] = np.log1p(pd.to_numeric(df['LotArea'], errors='coerce')).fillna(0)
        if 'LotFrontage' in df.columns:
            df['LotFrontage_log'] = np.log1p(pd.to_numeric(df['LotFrontage'], errors='coerce')).fillna(0)
            
        # Macro real estate combinations
        df['OverallGrade'] = (pd.to_numeric(df['OverallQual'], errors='coerce').fillna(5) ** 2) + pd.to_numeric(df['OverallCond'], errors='coerce').fillna(5)
        df['Age'] = pd.to_numeric(df['YrSold'], errors='coerce').fillna(2010) - pd.to_numeric(df['YearBuilt'], errors='coerce').fillna(2000)
        df['AvgRoomSize'] = pd.to_numeric(df['GrLivArea'], errors='coerce').fillna(1000) / pd.to_numeric(df['TotRmsAbvGrd'], errors='coerce').fillna(6)
        
        # Totaling up bathroom configurations (Half baths count as 0.5)
        df['TotalBathrooms'] = (
            pd.to_numeric(df['FullBath'], errors='coerce').fillna(0) + 
            (pd.to_numeric(df['HalfBath'], errors='coerce').fillna(0) * 0.5) + 
            pd.to_numeric(df['BsmtFullBath'], errors='coerce').fillna(0) + 
            (pd.to_numeric(df['BsmtHalfBath'], errors='coerce').fillna(0) * 0.5)
        )
        
        porch_cols = ['WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch']
        for col in porch_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)
        df['TotalOutdoorSF'] = df[porch_cols].sum(axis=1)
        
        # Binary checklist flags for asset presence
        df['Has_MasVnr'] = (pd.to_numeric(df['MasVnrArea'], errors='coerce').fillna(0) > 0).astype(int)
        df['Has_Basement'] = (pd.to_numeric(df['TotalBsmtSF'], errors='coerce').fillna(0) > 0).astype(int)
        df['Has2ndFloor'] = (pd.to_numeric(df['2ndFlrSF'], errors='coerce').fillna(0) > 0).astype(int)
        df['Has_OutdoorSpace'] = (df['TotalOutdoorSF'] > 0).astype(int)
        
        # High-homogeneity structural indicators
        df['Is_Functional'] = (df['Functional'].fillna('Typ').astype(str) == 'Typ').astype(int)
        df['Is_Normal_Proximity'] = (df['Condition1'].fillna('Norm').astype(str) == 'Norm').astype(int)
        df['Is_Hip_Roof'] = (df['RoofStyle'].fillna('Gable').astype(str) == 'Hip').astype(int)
        df['Is_Standard_Electrical'] = (df['Electrical'].fillna('SBrkr').astype(str) == 'SBrkr').astype(int)
        df['Is_Premium_MasVnr'] = df['MasVnrType'].fillna('None').astype(str).isin(['BrkFace', 'Stone']).astype(int)
        
        # Map quality ranking words straight to clean numerical scales
        qual_scale = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'NA': 0}
        fin_scale = {'GLQ': 6, 'ALQ': 5, 'BLQ': 4, 'Rec': 3, 'LwQ': 2, 'Unf': 1, 'NA': 0}
        
        ordinal_blueprints = {
            'ExterQual': qual_scale, 'BsmtQual': qual_scale, 'KitchenQual': qual_scale, 'HeatingQC': qual_scale,
            'BsmtFinType1': fin_scale, 'BsmtFinType2': fin_scale, 'FireplaceQu': qual_scale,
            'GarageFinish': {'Fin': 3, 'RFn': 2, 'Unf': 1, 'NA': 0},
            'PavedDrive': {'Y': 3, 'P': 2, 'N': 1},
            'LotShape': {'Reg': 1, 'IR1': 2, 'IR2': 3, 'IR3': 4},
            'LandSlope': {'Gtl': 1, 'Mod': 2, 'Sev': 3},
            'Fence': {'GdPrv': 4, 'MnPrv': 3, 'GdWo': 2, 'MnWw': 1, 'NA': 0}
        }
        
        for col, mapping in ordinal_blueprints.items():
            if col in df.columns:
                df[col] = df[col].fillna('NA').astype(str).map(mapping).fillna(0).astype(int)
                
        # Drop redundant columns to clear out noise
        drops = [
            'FullBath', 'HalfBath', 'BsmtFullBath', 'BsmtHalfBath', 'YearBuilt', 'YrSold',
            'TotRmsAbvGrd', 'KitchenAbvGr', 'GarageArea', 'GarageYrBlt', 'BsmtFinSF2',
            'LowQualFinSF', '2ndFlrSF', 'PoolArea', 'MiscVal', 'PoolQC', 'MiscFeature',
            'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch',
            'Utilities', 'LandContour', 'LotConfig', 'Condition2', 'Exterior2nd', 
            'RoofMatl', 'Heating', 'BsmtCond', 'GarageQual', 'GarageCond', 'Alley', 
            'Street', 'Functional', 'Condition1', 'RoofStyle', 'Electrical', 'MasVnrType'
        ]
        df = df.drop(columns=[c for c in drops if c in df.columns], errors='ignore')
        return df

In [18]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

median_cols = ['LotFrontage']
zero_fill_cols = ['BsmtFinSF1', 'TotalBsmtSF', 'BsmtUnfSF', 'MasVnrArea']
mode_cols = ['MSZoning', 'Exterior1st', 'SaleType', 'KitchenQual', 'GarageType', 'BsmtExposure', 'Foundation', 'Neighborhood', 'BldgType', 'HouseStyle', 'SaleCondition']

# Base cleaning step to handle missing blocks before doing math
imputer_transformer = ColumnTransformer(transformers=[
    ('median_imp', SimpleImputer(strategy='median'), median_cols),
    ('zero_imp', SimpleImputer(strategy='constant', fill_value=0.0), zero_fill_cols),
    ('mode_imp', SimpleImputer(strategy='most_frequent'), mode_cols)
], remainder='passthrough', verbose_feature_names_out=False)

# Build a clean pipeline
preprocessor = Pipeline(steps=[
    ('initial_imputation', imputer_transformer),
    ('feature_engineering', RealEstateFeatureEngineer())
])

print("🛠️ Baseline cleaning pipeline configured successfully!")

🛠️ Baseline cleaning pipeline configured successfully!


In [19]:
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.linear_model import Ridge

# 1. FIX: Fit the pipeline state explicitly first, then transform both sets
preprocessor.fit(X_train)
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# 2. Combine sets temporarily to ensure dummy columns line up perfectly
X_train_processed['is_train'] = 1
X_test_processed['is_train'] = 0
combined_sets = pd.concat([X_train_processed, X_test_processed], axis=0, ignore_index=True)

# 3. Cleanly convert all text columns to simple 0 or 1 numeric columns
combined_encoded = pd.get_dummies(combined_sets, drop_first=True, dtype=int)

# 4. Separate training data back out for modeling
X_train_final = combined_encoded[combined_encoded['is_train'] == 1].drop(columns=['is_train'])
X_test_final = combined_encoded[combined_encoded['is_train'] == 0].drop(columns=['is_train'])

# Ensure all remaining data columns are forced to flat float values
X_train_final = X_train_final.apply(pd.to_numeric, errors='coerce').fillna(0.0)
X_test_final = X_test_final.apply(pd.to_numeric, errors='coerce').fillna(0.0)

# 5. Run the Ridge Alpha Grid Search Optimization loop
kf_tune = KFold(n_splits=5, shuffle=True, random_state=42)
ridge_tune = Ridge()
param_grid = {'alpha': [0.1, 1.0, 5.0, 10.0, 15.0, 20.0, 50.0, 100.0]}

grid_search = GridSearchCV(
    estimator=ridge_tune,
    param_grid=param_grid,
    scoring='neg_root_mean_squared_error',
    cv=kf_tune,
    n_jobs=-1
)

grid_search.fit(X_train_final, y_train)
best_alpha = grid_search.best_params_['alpha']

print("📈 GRID SEARCH TUNING RESULTS:")
print("-" * 40)
print(f"Best Ridge Alpha Parameter Found : {best_alpha}")
print(f"Optimized Ridge Validation RMSE  : {-grid_search.best_score_:.5f}")

📈 GRID SEARCH TUNING RESULTS:
----------------------------------------
Best Ridge Alpha Parameter Found : 10.0
Optimized Ridge Validation RMSE  : 0.11204


In [21]:
import xgboost as xgb
import lightgbm as lgb
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error
import numpy as np

# 1. Initialize our three models using your newly optimized Ridge Alpha
models = {
    "Ridge": Ridge(alpha=best_alpha),
    "XGBoost": xgb.XGBRegressor(n_estimators=1000, learning_rate=0.03, max_depth=4, subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1),
    "LightGBM": lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.03, max_depth=4, subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1, verbose=-1)
}

kf_eval = KFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = {name: np.zeros(len(X_train_final)) for name in models}
test_preds = {name: np.zeros(len(X_test_final)) for name in models}

print("Running Cross-Validation..")
print("-" * 40)

# 2. Run the evaluation loop safely across all models
for name, model in models.items():
    cv_scores = []
    for train_idx, val_idx in kf_eval.split(X_train_final):
        # Slice our pre-processed, fully numeric data
        X_tr, y_tr = X_train_final.iloc[train_idx], y_train.iloc[train_idx]
        X_va, y_va = X_train_final.iloc[val_idx], y_train.iloc[val_idx]
        
        # Scale features per fold to keep valuations mathematically sound
        scaler = StandardScaler()
        X_tr_scaled = scaler.fit_transform(X_tr)
        X_va_scaled = scaler.transform(X_va)
        X_te_scaled = scaler.transform(X_test_final)
        
        model.fit(X_tr_scaled, y_tr)
        
        val_preds = model.predict(X_va_scaled)
        oof_preds[name][val_idx] = val_preds
        cv_scores.append(root_mean_squared_error(y_va, val_preds))
        
        # Build test predictions cumulatively across the 5 folds
        test_preds[name] += model.predict(X_te_scaled) / 5
        
    print(f"✔️ {name.ljust(15)} Mean CV RMSE: {np.mean(cv_scores):.5f}")

# 3. The Magic Blend (Applying our 60 / 25 / 15 weights)
final_oof_blend = (0.60 * oof_preds["Ridge"]) + (0.25 * oof_preds["XGBoost"]) + (0.15 * oof_preds["LightGBM"])
final_score = root_mean_squared_error(y_train, final_oof_blend)

print("-" * 40)
print(f"Final Blended Ensemble Validation RMSE: {final_score:.5f}")
print("-" * 40)

final_test_blend = (0.60 * test_preds["Ridge"]) + (0.25 * test_preds["XGBoost"]) + (0.15 * test_preds["LightGBM"])
final_prices = np.expm1(final_test_blend)

submission = pd.DataFrame({"Id": submission_ids, "SalePrice": final_prices})
submission.to_csv("ensemble_submission.csv", index=False)
print("💾 'ensemble_submission.csv' generated successfully!")

Running Cross-Validation..
----------------------------------------
✔️ Ridge           Mean CV RMSE: 0.11272
✔️ XGBoost         Mean CV RMSE: 0.11943
✔️ LightGBM        Mean CV RMSE: 0.12198
----------------------------------------
Final Blended Ensemble Validation RMSE: 0.11006
----------------------------------------
💾 'ensemble_submission.csv' generated successfully!
